In [17]:
import csv
import time
import random
import json
import html
from pathlib import Path
from typing import Dict, Tuple, Optional
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode
from concurrent.futures import ThreadPoolExecutor

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

from ftfy import fix_text

IN_PATH = Path("data/interim/gdelt_event_context_daily/2025/12/01/zambia_events_collapsed.csv")
OUT_PATH = IN_PATH.with_name(IN_PATH.stem + "_enriched.csv")

CACHE_PATH = IN_PATH.with_name(IN_PATH.stem + "_url_title_desc_cache.csv")

USER_AGENT = "Mozilla/5.0 (compatible; CCML/1.0)"
HEADERS = {"User-Agent": USER_AGENT, "Accept": "text/html,application/xhtml+xml"}

TIMEOUT_S = 20
MAX_RETRIES = 2
SLEEP_BETWEEN_REQ = (0.05, 0.15)
MAX_WORKERS = 20

MAX_TITLE_CHARS = 300
MAX_DESC_CHARS = 800

CACHE_FIELDS = ["url_normalized", "title", "description", "http_status", "fetch_error"]


def normalize_url(url: str) -> str:
    url = (url or "").strip()
    if not url:
        return url
    try:
        p = urlparse(url)
        q = [
            (k, v)
            for k, v in parse_qsl(p.query, keep_blank_values=True)
            if k.lower() not in {"gclid", "fbclid", "mc_cid", "mc_eid", "igshid", "spm", "ref", "ref_src"}
            and not any(k.lower().startswith(pref) for pref in ("utm_",))
        ]
        return urlunparse((p.scheme or "http", p.netloc.lower(), p.path, p.params, urlencode(q, doseq=True), ""))
    except Exception:
        return url

def truncate(s: Optional[str], n: int) -> str:
    s = (s or "").strip()
    return s[:n] if len(s) > n else s


def clean_meta_str(x: Optional[str]) -> str:
    """HTML unescape + fix mojibake + normalize whitespace."""
    if not x:
        return ""
    s = html.unescape(str(x))
    s = fix_text(s)  # fixes common encoding issues
    s = " ".join(s.split())
    return s


def _meta_content(soup: BeautifulSoup, *, name: str = "", prop: str = "") -> str:
    tag = None
    if name:
        tag = soup.find("meta", attrs={"name": name})
    if not tag and prop:
        tag = soup.find("meta", attrs={"property": prop})
    if tag and tag.get("content"):
        return tag["content"].strip()
    return ""


def fetch_title_desc(url: str, session: requests.Session) -> Tuple[str, str, int, str]:
    """
    Returns (title, description, http_status, fetch_error)
    """
    last_err = ""
    for attempt in range(MAX_RETRIES + 1):
        try:
            resp = session.get(url, timeout=TIMEOUT_S, allow_redirects=True)
            status = resp.status_code

            if status != 200 or not resp.text:
                if status in (429, 500, 502, 503, 504) and attempt < MAX_RETRIES:
                    time.sleep(0.5 * (attempt + 1))
                    continue
                return "", "", status, f"bad_status:{status}"

            soup = BeautifulSoup(resp.text, "html.parser")

            # Prefer OG title if present, else <title>
            title = _meta_content(soup, prop="og:title")
            if not title:
                title = soup.title.string if soup.title and soup.title.string else ""
            title = truncate(clean_meta_str(title), MAX_TITLE_CHARS)

            # Prefer meta description, fallback og:description
            desc = _meta_content(soup, name="description")
            if not desc:
                desc = _meta_content(soup, prop="og:description")
            desc = truncate(clean_meta_str(desc), MAX_DESC_CHARS)

            return title, desc, status, ""
        except Exception as e:
            last_err = f"exception:{type(e).__name__}:{str(e)[:200]}"
            if attempt < MAX_RETRIES:
                time.sleep(0.5 * (attempt + 1))
                continue
            return "", "", 0, last_err

    return "", "", 0, last_err or "failed_after_retries"


def load_cache() -> Dict[str, Dict[str, str]]:
    cache: Dict[str, Dict[str, str]] = {}
    if CACHE_PATH.exists():
        with open(CACHE_PATH, "r", encoding="utf-8") as f:
            for r in csv.DictReader(f):
                u = (r.get("url_normalized") or "").strip()
                if u:
                    cache[u] = r
    return cache


def append_cache_row(row: Dict[str, str]) -> None:
    write_header = not CACHE_PATH.exists()
    with open(CACHE_PATH, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CACHE_FIELDS)
        if write_header:
            w.writeheader()
        w.writerow({k: row.get(k, "") for k in CACHE_FIELDS})


def process_row(row: Dict[str, str], cache: Dict[str, Dict[str, str]], session: requests.Session) -> Dict[str, str]:
    url = (row.get("sourceurl") or "").strip()
    url_norm = normalize_url(url)
    row["url_normalized"] = url_norm

    # Non-http
    if not url_norm.startswith("http"):
        row.update({"title": "", "description": "", "http_status": "", "fetch_error": "non_http"})
        return row

    # Cache hit (only if it was actually useful)
    if url_norm in cache:
        c = cache[url_norm]
        c_title = (c.get("title") or "").strip()
        c_desc = (c.get("description") or "").strip()
        c_err  = (c.get("fetch_error") or "").strip()

        if (c_title or c_desc) and not c_err:
            row.update({
                "title": c_title,
                "description": c_desc,
                "http_status": str(c.get("http_status", "")),
                "fetch_error": ""
            })
            return row
        # else: ignore bad cache entry and refetch

    # Fetch with jitter (reduce 429)
    time.sleep(random.uniform(*SLEEP_BETWEEN_REQ))
    title, desc, status, err = fetch_title_desc(url_norm, session)

    row.update({
        "title": title,
        "description": desc,
        "http_status": str(status),
        "fetch_error": err
    })

    # Only cache successful fetches (or at least only cache 200 + some content)
    if not err and status == 200 and (title or desc):
        append_cache_row({
            "url_normalized": url_norm,
            "title": title,
            "description": desc,
            "http_status": str(status),
            "fetch_error": ""
        })

    return row


# -----------------------------
# MAIN
# -----------------------------
def enrich_file(in_path: Path) -> None:
    if not in_path.exists():
        raise FileNotFoundError(f"Input not found: {in_path}")

    cache = load_cache()

    with open(in_path, "r", newline="", encoding="utf-8") as f_in:
        rows = list(csv.DictReader(f_in))
        if not rows:
            print("No rows found.")
            return

    # Ensure we add these columns
    extra_cols = ["url_normalized", "title", "description", "http_status", "fetch_error"]
    fieldnames = list(dict.fromkeys(list(rows[0].keys()) + extra_cols))

    start = time.time()

    with requests.Session() as session:
        session.headers.update(HEADERS)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            results = list(
                tqdm(
                    ex.map(lambda r: process_row(r, cache, session), rows),
                    total=len(rows),
                    desc=f"Enriching {in_path.name}"
                )
            )

    with open(OUT_PATH, "w", newline="", encoding="utf-8") as f_out:
        w = csv.DictWriter(f_out, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(results)

    elapsed = time.time() - start
    print(f"\nSaved: {OUT_PATH}")
    print(f"Cache: {CACHE_PATH}")
    print(f"Time: {elapsed:.2f}s  |  Rows: {len(rows):,}")



In [18]:
enrich_file(IN_PATH)

Enriching zambia_events_collapsed.csv: 100%|██████████| 33/33 [00:08<00:00,  3.74it/s]


Saved: data/interim/gdelt_event_context_daily/2025/12/01/zambia_events_collapsed_enriched.csv
Cache: data/interim/gdelt_event_context_daily/2025/12/01/zambia_events_collapsed_url_title_desc_cache.csv
Time: 8.84s  |  Rows: 33
